In [0]:
import plotly.express as px

In [0]:
ambiente = 'project'

EXTREME_THRESHOLD = 0.9
PORCENTAJE_SAMPLE_DATA = 1
EVERY_N_YEARS = 20
RANDOM_SEED = 0
TEST_SIZE = 0.25

SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wave_energy', 'wave_power_kW_m']


In [0]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

df = data.toPandas()

In [0]:
limites = {
    'wave_energy': df['wave_energy'].quantile(EXTREME_THRESHOLD),
    'wave_power_kW_m': df['wave_power_kW_m'].quantile(EXTREME_THRESHOLD)
}
limites

In [0]:
extreme_threshold_label = f'{EXTREME_THRESHOLD}'.replace('.', '_')
df[f'wave_energy_{extreme_threshold_label}'] = df['wave_energy'].apply(lambda x: x >= limites['wave_energy'])
df[f'wave_power_kW_m_{extreme_threshold_label}'] = df['wave_power_kW_m'].apply(lambda x: x >= limites['wave_power_kW_m'])
df[f'wave_energy_power_{extreme_threshold_label}'] = df[f'wave_energy_{extreme_threshold_label}'] & df[f'wave_power_kW_m_{extreme_threshold_label}']

In [0]:
def graficar(X, color):
    fig = px.scatter_3d(
        X,
        x='wind_speed_ms',
        y='wave_period_s',
        z='wave_height_m',
        color=color,
        opacity=0.5
    )
    fig.update_traces(marker_size=3)
    fig.update_layout(
        scene=dict(
            aspectmode='cube'
        )
    )

    return fig